# Week 22 Optional Deep-Dive: Three Ways to Make the Retrain Loop Safer

You finished the main Week 22 notebook. The closing loop works end to end:
Week 20 drift detector triggers an Airflow DAG, the DAG kicks off Week 19
SageMaker training, registers a new model version, and updates the
`fraud-classifier-endpoint` that the Week 16 Strands supervisor calls.

This notebook is OPTIONAL. It does not introduce new infrastructure. It
asks three harder questions about what you just built:

1. We used a single drift test in Week 20. Was it the right test? When
   would a different test catch what we missed?
2. We updated the endpoint in place. What if the new model is WORSE than
   the old one? Can we test the new model on real traffic without risk?
3. Our DAG runs once per trigger. What happens when it runs twice
   because an operator clicked "Trigger DAG" by accident?

Each section is short: a concept, a small code experiment, and a "when
would you actually use this at Bread Financial" reflection.

Run this in Databricks or on your laptop with `AWS_PROFILE=di-mfa`. You do
NOT need to start any new SageMaker training jobs - we only call read-only
APIs and run small numpy / scipy computations.

## 1. Which drift test should we actually use?

In Week 20 we used a single test on `amount` and a handful of categorical
columns. That works, but it hides a real choice. Three tests dominate
industry practice for tabular drift:

- **PSI (Population Stability Index)**: bins the variable, compares bin
  frequencies between reference and current windows. Low sensitivity, very
  stable across sample sizes, interpretable thresholds (the 0.1 / 0.25
  rule of thumb credit-risk teams use). Works on numeric and categorical.
- **KS test (Kolmogorov-Smirnov)**: nonparametric test on the empirical
  CDF of a continuous variable. Reports a p-value. Sensitivity scales
  with sample size - on a 10M-row window it flags drift that nobody
  cares about.
- **Wasserstein distance (Earth Mover's distance)**: measures the cost
  of transporting one distribution to look like the other. Sensitive to
  mean shifts. Does NOT work on categorical features. Useful when
  distributions barely overlap (which KS handles poorly).

Bread Financial context: PSI dominates credit-risk MLOps because
regulators are familiar with it and its thresholds are stable. KS is the
default in scipy and many tutorials but is brittle at scale. Wasserstein
is the right answer when you care about HOW MUCH a distribution moved
(say, transaction amounts trending up over the holidays), not just
WHETHER it moved.

In [ ]:
# What this shows: same reference vs current sample, three drift tests,
# three different verdicts. Demonstrates why "I ran a drift test" is not
# a complete answer - the test you pick changes the answer.

import numpy as np
from scipy import stats

rng = np.random.default_rng(seed=42)

# Reference: last month's transaction amounts (log-normal, mean ~ $80)
reference = rng.lognormal(mean=4.3, sigma=0.6, size=10_000)

# Current: same shape but the mean shifted up ~12 percent (holiday season).
# A real drift, but a SMALL one.
current = rng.lognormal(mean=4.42, sigma=0.6, size=10_000)

# --- KS test ---
ks_stat, ks_pvalue = stats.ks_2samp(reference, current)
print(f"KS statistic = {ks_stat:.4f}, p-value = {ks_pvalue:.6f}")
print(f"  KS verdict at alpha=0.05: {'DRIFT' if ks_pvalue < 0.05 else 'no drift'}")

# --- Wasserstein distance ---
wd = stats.wasserstein_distance(reference, current)
print(f"Wasserstein distance = {wd:.4f} (units: dollars)")

# --- PSI (binned) ---
def psi(ref, cur, n_bins=10):
    # Use reference quantiles as bin edges, so the test is stable.
    edges = np.quantile(ref, np.linspace(0, 1, n_bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    ref_pct = np.histogram(ref, bins=edges)[0] / len(ref)
    cur_pct = np.histogram(cur, bins=edges)[0] / len(cur)
    # Floor to avoid log(0).
    ref_pct = np.clip(ref_pct, 1e-6, None)
    cur_pct = np.clip(cur_pct, 1e-6, None)
    return float(np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct)))

psi_value = psi(reference, current)
print(f"PSI = {psi_value:.4f}")
print(f"  PSI verdict (industry rule): "
      f"{'major drift' if psi_value > 0.25 else 'minor drift' if psi_value > 0.1 else 'no drift'}")

### Think about it

Run the cell again with `size=200_000` instead of `10_000`. The KS p-value
collapses to ~0; PSI barely moves; Wasserstein is unchanged. This is the
sample-size problem: at scale, KS flags everything.

For our Week 22 DAG, we want a test that ONLY fires on operationally
meaningful drift, because every trigger costs us a SageMaker training
job (~$2-3 each, see Section 3). PSI's stable thresholds make it the
right default for the retrain trigger. Save KS for ad-hoc investigation
in a notebook, and Wasserstein for the question "by how much did this
move".

What would you change in the Week 20 drift detector if you knew the
operator was going to look at the dashboard at 9am every weekday and
manually approve / reject the retrain? (Hint: the test does not have to
be the trigger.)

## 2. What if the new model is WORSE? The shadow variant pattern

Our DAG today updates `fraud-classifier-endpoint` with the new model
version using `UpdateEndpoint`. With SageMaker's `BlueGreenUpdatePolicy`,
that is already safer than a rolling update: SageMaker spins up the new
fleet beside the old one, shifts traffic per `TrafficRoutingConfiguration`,
and rolls back if a CloudWatch alarm fires.

But blue/green only protects against INFRASTRUCTURE failures - the new
container crashes, latency spikes, 5xx errors climb. It does NOT protect
against the new model being WRONG. A retrained model with a subtle bug
(say, a feature scaler from a stale fit) can serve 200 OK responses all
day with terrible predictions.

The **shadow variant** pattern fixes that. You deploy the new model as
a `ShadowProductionVariant` on the same endpoint. SageMaker REPLICATES
every production request to the shadow, captures both responses, and
returns ONLY the production response to the client. The shadow's output
never reaches the user. You collect those captured pairs to S3 and run
offline analysis - prediction agreement, calibration drift, fairness
metrics - BEFORE promoting.

Constraint: max one production variant + one shadow variant per
endpoint. `InitialVariantWeight` on the shadow is a placeholder; all
traffic still goes to production and is mirrored.

Cost: roughly 2x the endpoint compute for the shadow window (typically
24-72 hours). For a critical fraud endpoint, that is well-spent.

In [ ]:
# What this shows: the exact create_endpoint_config payload that turns
# a vanilla endpoint into a shadow-test endpoint. This is illustrative -
# we do NOT call create_endpoint here, because we already own
# fraud-classifier-endpoint from Week 19. Read it; do not run it against
# a live endpoint without instructor approval.

import boto3

sm = boto3.client("sagemaker", region_name="us-east-1")

# Names you would already have from Week 19 (registry approval step) and
# from your new retrain run.
CURRENT_MODEL_NAME = "fraud-classifier-v3"    # what is live today
CANDIDATE_MODEL_NAME = "fraud-classifier-v4"  # the just-trained candidate

shadow_config_payload = {
    "EndpointConfigName": "fraud-classifier-shadow-cfg-v4",
    "ProductionVariants": [
        {
            "VariantName": "production",
            "ModelName": CURRENT_MODEL_NAME,
            "InstanceType": "ml.m5.large",
            "InitialInstanceCount": 1,
            "InitialVariantWeight": 1.0,
        }
    ],
    "ShadowProductionVariants": [
        {
            "VariantName": "shadow-v4",
            "ModelName": CANDIDATE_MODEL_NAME,
            "InstanceType": "ml.m5.large",
            "InitialInstanceCount": 1,
            # This weight is a placeholder per SageMaker docs - all real
            # traffic still goes to production; shadow gets a mirror copy.
            "InitialVariantWeight": 1.0,
        }
    ],
    # DataCaptureConfig is what makes this pattern useful - both prod
    # and shadow responses land in S3 for offline comparison.
    "DataCaptureConfig": {
        "EnableCapture": True,
        "InitialSamplingPercentage": 100,
        "DestinationS3Uri": "s3://bread-academy-fraud/shadow-capture/v4/",
        "CaptureOptions": [
            {"CaptureMode": "Input"},
            {"CaptureMode": "Output"},
        ],
    },
}

print("Payload ready. To activate, you would call:")
print("  sm.create_endpoint_config(**shadow_config_payload)")
print("  sm.update_endpoint(EndpointName='fraud-classifier-endpoint',")
print("                     EndpointConfigName='fraud-classifier-shadow-cfg-v4')")
print()
print("DO NOT run those calls during class - they cost ~$0.10/hour extra")
print("for the shadow instance while the shadow window is open.")

### Think about it

In the Week 22 DAG we added a `validate_model` task gate before
`update_endpoint`. That gate runs an OFFLINE evaluation on a held-out
slice. Shadow testing runs an ONLINE evaluation on real production
traffic without exposing users to the new model.

Which would catch a stale-feature-scaler bug? (Hint: only one of them
sees real live distributions.) Which would catch a training-time bug
that hurts a subgroup? (Hint: only one of them runs on the offline
slice you specifically engineered to contain that subgroup.)

The honest answer at Bread Financial is "both", with shadow used for
the highest-risk endpoints and the offline gate used everywhere. For the
fraud endpoint we built, shadow makes sense. For an internal scoring
model used by 5 analysts, the offline gate alone is probably enough.

## 3. Three SageMaker calls that break when you re-run the DAG

Airflow DAGs get re-run. An operator clicks "Trigger DAG" twice. The
scheduler retries a task after a transient SageMaker throttle. A
catchup window fires three intervals at once. If any task in your DAG
is not IDEMPOTENT (safe to run again with the same input), you get
weird failures or, worse, weird silent successes.

Three SageMaker calls in our retrain DAG are NOT naturally idempotent:

1. `create_training_job(TrainingJobName=...)` raises `ResourceInUse` if
   the name already exists. Naive fix: append `datetime.utcnow()`. Then
   the SECOND run of the DAG starts a SECOND training job, doubling
   cost.
2. `create_model_package(...)` registers a new version every call.
   Re-running creates duplicate v4, v5, v6 packages with identical
   artifacts.
3. `update_endpoint(...)` is mostly safe BUT will fail with
   `ValidationException` if the endpoint is already in `Updating`
   status from a previous run. The DAG task crashes and you cannot
   tell if the previous update succeeded.

The fix is the same pattern for all three: **make the name a
deterministic function of the trigger** (the drift detection timestamp,
the data version, the input artifact hash), and **check before you
create**. If the resource already exists in a terminal-success state,
no-op. If it exists in a failed state, optionally retry. If it exists
in a still-running state, wait.

In [ ]:
# What this shows: a small wrapper that makes create_training_job
# rerun-safe. Same input -> same name -> same outcome, regardless of
# how many times Airflow calls the task.

import hashlib
import boto3
from botocore.exceptions import ClientError

sm = boto3.client("sagemaker", region_name="us-east-1")

def deterministic_job_name(prefix: str, drift_window_iso: str, data_uri: str) -> str:
    # The trigger is uniquely identified by the drift window and the
    # input data version. Hash them so the name is stable across reruns.
    sig = hashlib.sha256(f"{drift_window_iso}|{data_uri}".encode()).hexdigest()[:12]
    # SageMaker job names: max 63 chars, [a-zA-Z0-9-].
    return f"{prefix}-{sig}"

def create_or_wait_training_job(name: str, payload: dict) -> str:
    # Returns one of: "Completed", "Failed", "Stopped", "InProgress".
    try:
        sm.create_training_job(TrainingJobName=name, **payload)
        print(f"Created new training job {name}")
    except ClientError as e:
        code = e.response["Error"]["Code"]
        # "ResourceInUse" means a job with this name already exists -
        # which is EXACTLY what we want on a rerun.
        if code != "ResourceInUse":
            raise
        print(f"Training job {name} already exists, reusing it")
    # Either way, describe it to learn the current status.
    desc = sm.describe_training_job(TrainingJobName=name)
    return desc["TrainingJobStatus"]

# Illustration of the call site - do NOT run this against real SageMaker
# during class.
job_name = deterministic_job_name(
    prefix="fraud-retrain",
    drift_window_iso="2026-05-19T08:00:00Z",
    data_uri="s3://bread-academy-fraud/curated/2026-05-19/",
)
print(f"Stable name for this trigger: {job_name}")
print("Re-running the DAG with the same drift window will reuse this job,")
print("not create a duplicate. That is the whole point.")

## Where to go from here

You looked at three orthogonal ways to harden the closing loop:

- A better drift test changes WHICH triggers fire.
- A shadow variant changes WHAT we learn before we promote.
- Idempotent task names change WHAT HAPPENS when the DAG re-runs.

None of these required new infrastructure. They are all small changes to
code we already wrote in Weeks 19-22.

Topics intentionally NOT covered here, because Week 24 capstone will:

- Airflow Celery / KEDA executors and autoscaling worker pools.
- AutoDAG / dynamic task mapping for per-segment retrains.
- Cost-budgeted backfill strategies for the historical data lake.

If you want to read further BEFORE Week 24, the AWS shadow-testing blog
post and the SageMaker model-lineage notebook example (both linked
below) are the two best deep dives.

## Sources verified (this session, May 2026)

- https://docs.aws.amazon.com/sagemaker/latest/dg/model-shadow-deployment.html
- https://aws.amazon.com/blogs/machine-learning/minimize-the-production-impact-of-ml-model-updates-with-amazon-sagemaker-shadow-testing/
- https://docs.aws.amazon.com/sagemaker/latest/APIReference/API_CreateEndpointConfig.html
- https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/sagemaker/client/update_endpoint.html
- https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/sagemaker/client/create_training_job.html
- https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/sagemaker/client/query_lineage.html
- https://docs.aws.amazon.com/sagemaker/latest/dg/querying-lineage-entities.html
- https://www.evidentlyai.com/blog/data-drift-detection-large-datasets
- https://nannyml.readthedocs.io/en/stable/how_it_works/univariate_drift_comparison.html
- https://superwise.ai/blog/a-hands-on-introduction-to-drift-metrics/